In [ ]:
# CMS Quality Ratings and Enrollee Experience Analysis

## Project Question
Are stronger medical-care quality ratings associated with better enrollee-reported health plan experiences?

## Data Sources
This analysis uses publicly available CMS Marketplace data:

- PY2026 Nationwide Quality Rating System Public Use File
- 2025 Qualified Health Plan Enrollee Survey Public Use File

## Analytical Focus
The primary explanatory variables are the Medical Care Rating and Medical Care Score from the QRS dataset.

The primary target variable is the case-mix adjusted Health Plan Rating score from the Enrollee Survey.

## Workflow
1. Data intake and profiling
2. Data cleaning and validation
3. Dataset integration
4. Exploratory analysis
5. Statistical analysis
6. Visualization
7. Business interpretation

## 1. Data Intake and Profiling

Before cleaning or analyzing the data, the datasets are loaded and reviewed for dimensions, structure, column names, data types, duplicates, and missing values.

In [1]:
import pandas as pd
from pathlib import Path

# Set project paths
project_root = Path("..")
data_raw = project_root / "data_raw"
data_clean = project_root / "data_clean"

# File paths
qrs_file = data_raw / "PY2026 - Nationwide PUF.csv"
survey_file = data_raw / "2025-QHPEnrolleeSurvey-PUF.xlsx"

In [2]:
qrs = pd.read_csv(qrs_file)
survey = pd.read_excel(survey_file)

In [3]:
print("QRS shape:", qrs.shape)
print("Survey shape:", survey.shape)

display(qrs.head())
display(survey.head())

QRS shape: (331, 160)
Survey shape: (354, 423)


,Issuer_Id,State,Plan_Type,Global_rate,Global_score,S1_rate,S1_score,S2_rate,S2_score,S3_rate,...,S3M43_rate,S3M43_score,S3M44_den,S3M44_elig,S3M44_rate,S3M44_score,S3M45_den,S3M45_elig,S3M45_rate,S3M45_score
0,10091,OR,PPO,4,81.860,3,77.383,4,88.955,5,...,55.323009,87.750,108,1288,76.224063,97.788,148,1288,74.034084,90.068
1,10207,VA,HMO,5,90.341,5,91.638,5,95.263,4,...,48.132765,76.345,120,2329,65.420631,83.929,184,2329,74.358840,90.463
2,11177,NY,HMO,4,83.801,4,83.349,CSR-I,CSR-I,4,...,45.328577,NC,79,1278,64.440988,NC,94,1278,67.210163,NC
3,11269,WY,PPO,2,67.476,2,64.036,CSR-I,CSR-I,4,...,46.278278,NC,72,1296,73.543534,NC,112,1296,76.582585,93.168
4,11512,NC,EPO,4,83.318,3,79.517,5,95.010,4,...,46.294328,73.429,138,3231,67.355652,86.411,201,3231,69.988291,85.146


,REPORTING_UNIT_ID,N_COMPLETED_SURVEYS,RESPONSE_RATE,ACCESSINFO_SCORE_UNADJ,ACCESSINFO_SCORE_ADJ,ACCESSINFO_N_NS,ACCESSINFO_N_U,ACCESSINFO_N_A,ACCESSINFO_N_NA,ACCESSINFO_N_MISS,...,TESTS_SCORE_ADJ,TESTS_N_NS,TESTS_N_U,TESTS_N_A,TESTS_N_ASNA,TESTS_N_MISS,USE_TOBACCO_N_DK,USE_TOBACCO_N_ED_SD,USE_TOBACCO_N_MISS,USE_TOBACCO_N_NOT_AT_ALL
0,10091-OR-PPO,150,20.1%,55.01,55.32,50.0,39.0,41.0,20.0,0.0,...,88.62,NaN,20.0,87.0,34.0,NaN,NaN,NaN,NaN,145
1,10207-VA-HMO,187,15.7%,46.85,48.13,89.0,44.0,35.0,NaN,NaN,...,89.13,NaN,34.0,100.0,43.0,NaN,NaN,NaN,NaN,179
2,11177-NY-HMO,98,16.1%,45.11,45.33,47.0,19.0,20.0,NaN,NaN,...,82.99,NaN,13.0,49.0,19.0,NaN,0.0,NaN,NaN,84
3,11269-WY-PPO,117,16.7%,45.99,46.28,47.0,26.0,24.0,20.0,0.0,...,87.27,NaN,NaN,55.0,44.0,NaN,0.0,NaN,NaN,110
4,11512-NC-EPO,204,13.4%,46.08,46.29,91.0,55.0,34.0,NaN,NaN,...,90.19,NaN,31.0,111.0,49.0,NaN,NaN,15.0,NaN,180


### Initial Intake Notes

The datasets were successfully loaded and their dimensions and first records were reviewed to confirm structure before any cleaning or transformation.

Further profiling will examine:
- column names
- data types
- duplicate records
- missing values
- join fields

### Structure and Data Quality Review

The next step is to examine column names, data types, duplicate records, and missing values before creating any derived fields or joining the datasets.

In [4]:
# Review column names
print("QRS columns:")
print(qrs.columns.tolist())

print("\nSurvey columns:")
print(survey.columns.tolist())

QRS columns:
['Issuer_Id', 'State', 'Plan_Type', 'Global_rate', 'Global_score', 'S1_rate', 'S1_score', 'S2_rate', 'S2_score', 'S3_rate', 'S3_score', 'S1M58_den', 'S1M58_rate', 'S1M58_score', 'S1M50_den', 'S1M50_elig', 'S1M50_rate', 'S1M50_score', 'S1M2_den', 'S1M2_elig', 'S1M2_rate', 'S1M2_score', 'S1M49_den', 'S1M49_elig', 'S1M49_rate', 'S1M49_score', 'S1M59_den', 'S1M59_rate', 'S1M59_score', 'S1M17_den', 'S1M17_elig', 'S1M17_rate', 'S1M17_score', 'S1M51_den', 'S1M51_elig', 'S1M51_rate', 'S1M51_score', 'S1M53_den', 'S1M53_elig', 'S1M53_rate', 'S1M53_score', 'S1M23_den', 'S1M23_elig', 'S1M23_rate', 'S1M23_score', 'S1M6_den', 'S1M6_elig', 'S1M6_rate', 'S1M6_score', 'S1M9_den', 'S1M9_elig', 'S1M9_rate', 'S1M9_score', 'S1M52_den', 'S1M52_elig', 'S1M52_rate', 'S1M52_score', 'S1M56_den', 'S1M56_elig', 'S1M56_rate', 'S1M56_score', 'S1M47_den', 'S1M47_elig', 'S1M47_rate', 'S1M47_score', 'S1M55_den', 'S1M55_elig', 'S1M55_rate', 'S1M55_score', 'S1M48_den', 'S1M48_elig', 'S1M48_rate', 'S1M48_sco

In [5]:
# Review data types
print("QRS data types:")
display(qrs.dtypes)

print("\nSurvey data types:")
display(survey.dtypes)

QRS data types:


Issuer_Id         int64
State            object
Plan_Type        object
Global_rate      object
Global_score     object
                 ...   
S3M44_score      object
S3M45_den         int64
S3M45_elig        int64
S3M45_rate      float64
S3M45_score      object
Length: 160, dtype: object


Survey data types:


REPORTING_UNIT_ID            object
N_COMPLETED_SURVEYS          object
RESPONSE_RATE                object
ACCESSINFO_SCORE_UNADJ      float64
ACCESSINFO_SCORE_ADJ         object
                             ...   
TESTS_N_MISS                float64
USE_TOBACCO_N_DK            float64
USE_TOBACCO_N_ED_SD         float64
USE_TOBACCO_N_MISS          float64
USE_TOBACCO_N_NOT_AT_ALL      int64
Length: 423, dtype: object

In [6]:
# Check for duplicate rows
print("QRS duplicate rows:", qrs.duplicated().sum())
print("Survey duplicate rows:", survey.duplicated().sum())

# Check duplicate reporting units in the survey
print(
    "Survey duplicate reporting unit IDs:",
    survey["REPORTING_UNIT_ID"].duplicated().sum()
)

QRS duplicate rows: 0
Survey duplicate rows: 0
Survey duplicate reporting unit IDs: 0


In [7]:
# Missing values in key QRS variables
qrs_key_cols = [
    "Issuer_Id",
    "State",
    "Plan_Type",
    "S1_rate",
    "S1_score"
]

print("QRS key-variable missingness:")
display(qrs[qrs_key_cols].isnull().sum())

# Missing values in key survey variables
survey_key_cols = [
    "REPORTING_UNIT_ID",
    "RATE_HP_SCORE_ADJ"
]

print("Survey key-variable missingness:")
display(survey[survey_key_cols].isnull().sum())

QRS key-variable missingness:


Issuer_Id    0
State        0
Plan_Type    0
S1_rate      0
S1_score     0
dtype: int64

Survey key-variable missingness:


REPORTING_UNIT_ID    0
RATE_HP_SCORE_ADJ    0
dtype: int64

### Profiling Findings

Initial profiling found no duplicate records in either dataset and no duplicate
reporting-unit identifiers in the Enrollee Survey.

The primary QRS variables and primary survey target also contain no missing
values based on standard null-value checks.

However, several rating and score variables are stored as `object` rather than
numeric data types. Review of the dataset preview indicates that CMS uses
special text codes such as `CSR-I` and `NC` in some quality fields.

These values must be interpreted before numeric conversion. They should not be
automatically treated as missing data because they may represent meaningful CMS
reporting or eligibility statuses.

## 2. Key Variable Validation

Before transforming the datasets, the values contained in the primary analytical
variables are reviewed to identify numeric values and CMS-specific reporting codes.

In [8]:
# Review unique values in primary QRS variables
for col in ["S1_rate", "S1_score"]:
    print(f"\n{col}")
    print(qrs[col].value_counts(dropna=False).sort_index())


S1_rate
S1_rate
1         14
2         69
3        114
4         93
5         18
CSR-I     23
Name: count, dtype: int64

S1_score
S1_score
100.931     1
50.203      1
51.864      1
53.175      1
53.517      1
           ..
97.560      1
97.604      1
98.686      1
99.521      1
CSR-I      23
Name: count, Length: 308, dtype: int64


In [9]:
# Review the target variable
print("RATE_HP_SCORE_ADJ data type:")
print(survey["RATE_HP_SCORE_ADJ"].dtype)

print("\nSample values:")
display(survey["RATE_HP_SCORE_ADJ"].head(20))

print("\nUnique value count:")
print(survey["RATE_HP_SCORE_ADJ"].nunique())

RATE_HP_SCORE_ADJ data type:
object

Sample values:


0     74.03
1     74.36
2     67.21
3     76.58
4     69.99
5     70.98
6     69.31
7     73.43
8     81.86
9     62.33
10    68.88
11    70.82
12    71.05
13    75.39
14    76.77
15    67.70
16    62.51
17    66.28
18    68.77
19    70.06
Name: RATE_HP_SCORE_ADJ, dtype: object


Unique value count:
324


In [10]:
# Identify non-numeric values in the QRS variables
for col in ["S1_rate", "S1_score"]:
    numeric_version = pd.to_numeric(qrs[col], errors="coerce")

    non_numeric = qrs.loc[
        numeric_version.isna() & qrs[col].notna(),
        col
    ].unique()

    print(f"{col} non-numeric values:", non_numeric)

S1_rate non-numeric values: ['CSR-I']
S1_score non-numeric values: ['CSR-I']


In [11]:
# Check the survey target for non-numeric values
target_numeric = pd.to_numeric(
    survey["RATE_HP_SCORE_ADJ"],
    errors="coerce"
)

non_numeric_target = survey.loc[
    target_numeric.isna() & survey["RATE_HP_SCORE_ADJ"].notna(),
    "RATE_HP_SCORE_ADJ"
].unique()

print("RATE_HP_SCORE_ADJ non-numeric values:", non_numeric_target)

RATE_HP_SCORE_ADJ non-numeric values: ['     .']


### Key Variable Validation Findings

The Medical Care Rating (`S1_rate`) and Medical Care Score (`S1_score`)
contain the CMS code `CSR-I` for 23 reporting units.

According to the CMS data dictionary, `CSR-I` indicates insufficient data
to calculate a component score or rating under the QRS methodology. These
records are valid reporting units, but they do not contain usable Medical
Care performance values for the primary analysis.

The adjusted Health Plan Rating (`RATE_HP_SCORE_ADJ`) contains one
non-numeric value represented by a period (`.`). Review of the source data
shows that this value belongs to the `National` summary row rather than an
individual reporting unit.

Therefore, non-numeric values will be preserved in the raw datasets but
excluded from the analysis-ready dataset when numeric Medical Care and
enrollee-experience measures are required.

## 3. Data Cleaning and Preparation

Clean analytical copies of the datasets are created so that the original
CMS source data remains unchanged.

The primary quality and experience variables are converted to numeric
formats. CMS reporting codes that cannot be used as numerical measurements
are converted to missing values only within the analysis-ready copies.

In [12]:
# Create working copies and preserve the raw DataFrames
qrs_clean = qrs.copy()
survey_clean = survey.copy()

In [13]:
# Convert Medical Care variables to numeric
# CSR-I will become NaN in the clean analytical copy
qrs_clean["S1_rate"] = pd.to_numeric(
    qrs_clean["S1_rate"],
    errors="coerce"
)

qrs_clean["S1_score"] = pd.to_numeric(
    qrs_clean["S1_score"],
    errors="coerce"
)

In [15]:
# Convert adjusted health plan rating to numeric
survey_clean["RATE_HP_SCORE_ADJ"] = pd.to_numeric(
    survey_clean["RATE_HP_SCORE_ADJ"],
    errors="coerce"
)

In [16]:
print("QRS Medical Care Rating missing after conversion:",
      qrs_clean["S1_rate"].isna().sum())

print("QRS Medical Care Score missing after conversion:",
      qrs_clean["S1_score"].isna().sum())

print("Survey target missing after conversion:",
      survey_clean["RATE_HP_SCORE_ADJ"].isna().sum())

QRS Medical Care Rating missing after conversion: 23
QRS Medical Care Score missing after conversion: 23
Survey target missing after conversion: 1


In [17]:
# Construct reporting-unit identifier to match the survey dataset
qrs_clean["REPORTING_UNIT_ID"] = (
    qrs_clean["Issuer_Id"].astype(str)
    + "-"
    + qrs_clean["State"]
    + "-"
    + qrs_clean["Plan_Type"]
)

In [18]:
display(
    qrs_clean[
        ["Issuer_Id", "State", "Plan_Type", "REPORTING_UNIT_ID"]
    ].head()
)

,Issuer_Id,State,Plan_Type,REPORTING_UNIT_ID
0,10091,OR,PPO,10091-OR-PPO
1,10207,VA,HMO,10207-VA-HMO
2,11177,NY,HMO,11177-NY-HMO
3,11269,WY,PPO,11269-WY-PPO
4,11512,NC,EPO,11512-NC-EPO


### Cleaning Results

The Medical Care Rating and Medical Care Score each produced 23 missing values
after numeric conversion. These correspond to CMS `CSR-I` reporting codes,
which indicate that a numeric Medical Care rating or score was not available
for those reporting units.

The adjusted Health Plan Rating produced one missing value after conversion,
corresponding to the non-numeric national summary value.

The original source datasets remain unchanged. These transformations were
performed only in the analytical copies of the data.

A standardized `REPORTING_UNIT_ID` was also constructed in the QRS dataset
using issuer ID, state, and plan type to support validation and integration
with the Enrollee Survey.

## 4. Join Validation

Before merging the QRS and Enrollee Survey datasets, reporting-unit identifiers
are compared to determine how many records match across sources and whether
either dataset contains unmatched reporting units.

In [19]:
# Create sets of reporting-unit IDs
qrs_ids = set(qrs_clean["REPORTING_UNIT_ID"])
survey_ids = set(survey_clean["REPORTING_UNIT_ID"])

# Compare reporting-unit coverage
matched_ids = qrs_ids.intersection(survey_ids)
qrs_only_ids = qrs_ids.difference(survey_ids)
survey_only_ids = survey_ids.difference(qrs_ids)

print("QRS reporting units:", len(qrs_ids))
print("Survey reporting units:", len(survey_ids))
print("Matched reporting units:", len(matched_ids))
print("QRS-only reporting units:", len(qrs_only_ids))
print("Survey-only reporting units:", len(survey_only_ids))

QRS reporting units: 331
Survey reporting units: 354
Matched reporting units: 331
QRS-only reporting units: 0
Survey-only reporting units: 23


In [20]:
print("QRS-only IDs:")
print(sorted(qrs_only_ids))

print("\nSurvey-only IDs:")
print(sorted(survey_only_ids))

QRS-only IDs:
[]

Survey-only IDs:
['16842-FL-PPO', '26065-SC-HMO', '30252-FL-POS', '30751-MT-POS', '33602-TX-POS', '33764-SC-HMO', '37777-NJ-EPO', '38927-UT-HMO', '45334-GA-HMO', '48772-AR-POS', '51396-CA-HMO', '55584-CO-EPO', '64004-DE-EPO', '65428-NM-HMO', '72547-IL-PPO', '79475-WI-POS', '80180-WI-HMO', '84600-KS-EPO', '84867-OH-HMO', '90787-LA-EPO', '93909-PA-HMO', '96992-IN-HMO', 'National']


### Join Validation Findings

All 331 reporting units contained in the Nationwide QRS dataset have a
corresponding record in the Enrollee Survey dataset.

The Enrollee Survey contains 23 additional reporting units that are not
present in the Nationwide QRS file. These include the national summary row
and other survey reporting units without corresponding records in the
Nationwide QRS dataset.

Because the research question evaluates enrollee experience in relation to
QRS Medical Care performance, the QRS population will serve as the analytical
base. No QRS reporting units will be lost when the datasets are integrated.

## 5. Dataset Integration

The Nationwide QRS dataset is used as the analytical base because the
research question requires a Medical Care quality rating or score.

Only selected variables needed for the primary analysis and later
interpretation are brought forward from each source. The datasets are joined
using `REPORTING_UNIT_ID`.

A one-to-one relationship is required because each reporting unit should
appear only once in each source.

In [21]:
# Select QRS variables needed for analysis
qrs_analysis = qrs_clean[
    [
        "REPORTING_UNIT_ID",
        "Issuer_Id",
        "State",
        "Plan_Type",
        "Global_rate",
        "Global_score",
        "S1_rate",
        "S1_score",
        "S2_rate",
        "S2_score",
        "S3_rate",
        "S3_score"
    ]
].copy()

In [22]:
# Select Enrollee Survey variables needed for analysis
survey_analysis = survey_clean[
    [
        "REPORTING_UNIT_ID",
        "N_COMPLETED_SURVEYS",
        "RESPONSE_RATE",
        "RATE_HP_SCORE_ADJ",
        "RATE_HEALTH_CARE_SCORE_ADJ",
        "GETCAREQUICK_SCORE_ADJ",
        "GETNEEDEDCARE_SCORE_ADJ",
        "CUSTOMERSERVICE_SCORE_ADJ",
        "EXPERIENCEWITHCOST_SCORE_ADJ",
        "CARECOORD_SCORE_ADJ",
        "ACCESSINFO_SCORE_ADJ"
    ]
].copy()

In [23]:
analysis = qrs_analysis.merge(
    survey_analysis,
    on="REPORTING_UNIT_ID",
    how="left",
    validate="one_to_one",
    indicator=True
)

In [24]:
validate="one_to_one"

In [25]:
indicator=True

In [26]:
print("Merged shape:", analysis.shape)

print("\nMerge status:")
print(analysis["_merge"].value_counts())

display(analysis.head())

Merged shape: (331, 23)

Merge status:
_merge
both          331
left_only       0
right_only      0
Name: count, dtype: int64


,REPORTING_UNIT_ID,Issuer_Id,State,Plan_Type,Global_rate,Global_score,S1_rate,S1_score,S2_rate,S2_score,...,RESPONSE_RATE,RATE_HP_SCORE_ADJ,RATE_HEALTH_CARE_SCORE_ADJ,GETCAREQUICK_SCORE_ADJ,GETNEEDEDCARE_SCORE_ADJ,CUSTOMERSERVICE_SCORE_ADJ,EXPERIENCEWITHCOST_SCORE_ADJ,CARECOORD_SCORE_ADJ,ACCESSINFO_SCORE_ADJ,_merge
0,10091-OR-PPO,10091,OR,PPO,4,81.860,3.0,77.383,4,88.955,...,20.1%,74.03,78.31,61.23,63.77,80.62,77.66,82.60,55.32,both
1,10207-VA-HMO,10207,VA,HMO,5,90.341,5.0,91.638,5,95.263,...,15.7%,74.36,82.88,75.19,73.08,70.08,80.86,82.49,48.13,both
2,11177-NY-HMO,11177,NY,HMO,4,83.801,4.0,83.349,CSR-I,CSR-I,...,16.1%,67.21,77.55,69.28,63.51,70.29,73.72,81.67,45.33,both
3,11269-WY-PPO,11269,WY,PPO,2,67.476,2.0,64.036,CSR-I,CSR-I,...,16.7%,76.58,83.22,77.04,75.05,79.75,75.42,84.04,46.28,both
4,11512-NC-EPO,11512,NC,EPO,4,83.318,3.0,79.517,5,95.010,...,13.4%,69.99,80.13,76.21,75.22,70.18,73.18,81.95,46.29,both


In [27]:
print("Medical Care Rating missing:",
      analysis["S1_rate"].isna().sum())

print("Medical Care Score missing:",
      analysis["S1_score"].isna().sum())

print("Health Plan Experience missing:",
      analysis["RATE_HP_SCORE_ADJ"].isna().sum())

Medical Care Rating missing: 23
Medical Care Score missing: 23
Health Plan Experience missing: 0


### Integration Results

The QRS and Enrollee Survey datasets were successfully joined using
`REPORTING_UNIT_ID`.

All 331 QRS reporting units matched a corresponding Enrollee Survey record,
with no unmatched QRS records. The merged dataset contains 331 rows and
23 selected variables.

The adjusted Health Plan Rating is complete for all matched reporting units.
The Medical Care Rating and Medical Care Score contain 23 unavailable values,
corresponding to reporting units previously identified with the CMS `CSR-I`
status.

These records remain valid reporting units but cannot be included in analyses
that require a numeric Medical Care rating or score.

## 6. Analysis-Ready Dataset

The final analytical sample includes only reporting units with usable numeric
Medical Care ratings, Medical Care scores, and adjusted Health Plan Rating
scores.

Records with unavailable Medical Care performance are excluded from the primary
statistical analysis but remain preserved in the original and intermediate
datasets.

In [28]:
# Keep only records with usable values for the primary analysis
analysis_ready = analysis.dropna(
    subset=[
        "S1_rate",
        "S1_score",
        "RATE_HP_SCORE_ADJ"
    ]
).copy()

print("Merged reporting units:", len(analysis))
print("Analysis-ready reporting units:", len(analysis_ready))
print("Excluded reporting units:", len(analysis) - len(analysis_ready))

Merged reporting units: 331
Analysis-ready reporting units: 308
Excluded reporting units: 23


In [29]:
print("Medical Care rating distribution:")
print(
    analysis_ready["S1_rate"]
    .value_counts()
    .sort_index()
)

Medical Care rating distribution:
S1_rate
1.0     14
2.0     69
3.0    114
4.0     93
5.0     18
Name: count, dtype: int64


In [30]:
analysis_ready = analysis_ready.drop(columns="_merge")

In [31]:
output_file = data_clean / "cms_qrs_enrollee_experience_clean.csv"

analysis_ready.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)

Saved: ..\data_clean\cms_qrs_enrollee_experience_clean.csv


### Final Data Preparation Summary

The final analysis-ready dataset contains 308 Marketplace reporting units with
complete numeric values for:

- Medical Care Rating
- Medical Care Score
- Adjusted Health Plan Rating

Twenty-three reporting units were excluded from the primary analytical sample
because CMS did not calculate numeric Medical Care performance for those units.

The cleaned dataset was exported for use in exploratory and statistical analysis.
The original CMS source files remain unchanged.

## Notebook Conclusion

The CMS Nationwide QRS and QHP Enrollee Survey datasets were successfully
profiled, cleaned, validated, and integrated at the reporting-unit level.

The final analysis-ready dataset contains 308 reporting units with complete
numeric Medical Care performance and adjusted Health Plan Rating measures.

This cleaned dataset will be used in the next stage of the project to explore
whether stronger Medical Care performance is associated with better
enrollee-reported health plan experiences.